# Student Performance Prediction — Regression Pipeline

This notebook formulates **Student_Performance.csv** as a **Regression** problem predicting the continuous target variable **`Performance Index`** (10,000 rows, 6 columns), following the `Regression Models.ipynb` workflow and the structured notebook layout.


## 1. Importing Libraries

Import data processing, visualization, preprocessing, regression models, and evaluation metrics, with safe optional imports for `xgboost`, `catboost`, and `lightgbm`.


In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    BaggingRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor
)
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False


## 2. Exploratory Data Analysis (EDA)

Load `Student_Performance.csv` and inspect dimensions, preview rows, column data types, missing values, duplicates, and summary statistics.


In [ ]:
df = pd.read_csv('Student_Performance.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
df.describe(include='all')


### 2.1 Feature Correlations and Categorical Breakdown

Inspect linear correlations with `Performance Index` and frequency distributions for `Extracurricular Activities` and `Hours Studied`.


In [ ]:
print('Correlation with Performance Index:')
print(df.corr(numeric_only=True)['Performance Index'].sort_values(ascending=False))
print('\nExtracurricular Activities counts:')
print(df['Extracurricular Activities'].value_counts(dropna=False))


## 3. Data Visualization

Visualize distributions of numerical features and target `Performance Index`, along with relationships against `Previous Scores` and `Hours Studied`.


In [ ]:
num_plot_cols = ['Hours Studied', 'Previous Scores', 'Sleep Hours', 'Sample Question Papers Practiced', 'Performance Index']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, col in zip(axes.flat, num_plot_cols):
    sns.histplot(df[col].dropna(), bins=20, kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
axes.flat[-1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='Previous Scores', y='Performance Index', alpha=0.4, ax=axes[0])
axes[0].set_title('Previous Scores vs. Performance Index')
sns.boxplot(data=df, x='Extracurricular Activities', y='Performance Index', ax=axes[1])
axes[1].set_title('Performance Index by Extracurricular Activities')
plt.tight_layout()
plt.show()


## 4. Data Cleaning and Preparation

Prepare `data` from `Student_Performance.csv` and separate input features `X` and regression target `y = data[['Performance Index']]`.


In [ ]:
data = df.copy()
X = data.drop('Performance Index', axis=1)
y = data[['Performance Index']]
print('X shape:', X.shape, 'y shape:', y.shape)


## 5. Train/Test Split, Imputation, Encoding, and Scaling

Split the dataset 80/20 (`random_state=100`), impute missing values (`Sample Question Papers Practiced` and `Extracurricular Activities`) strictly on training data, encode categorical features with `OrdinalEncoder`, and scale features with `MinMaxScaler(feature_range=(0, 1))`.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=100)
print('X_train =', X_train.shape, 'y_train =', y_train.shape)
print('X_test =', X_test.shape, 'y_test =', y_test.shape)


In [ ]:
num_cols = ['Hours Studied', 'Previous Scores', 'Sleep Hours', 'Sample Question Papers Practiced']
cat_cols = ['Extracurricular Activities']
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])
my_scaler = MinMaxScaler(feature_range=(0, 1))
X_train = my_scaler.fit_transform(X_train)
X_test = my_scaler.transform(X_test)


## 6. Evaluation Metrics Definition

Define `regression_metrics` to compute `MAE`, `MSE`, `RMSE`, and `R2` (`r2_score`).


In [ ]:
def regression_metrics(model, X_data, y_actual):
    y_pred = model.predict(X_data)
    MAE = mean_absolute_error(y_actual, y_pred)
    MSE = mean_squared_error(y_actual, y_pred)
    RMSE = np.sqrt(MSE)
    R2 = r2_score(y_actual, y_pred)
    return MAE, MSE, RMSE, R2


## 7. Individual Regression Models Training and Evaluation

Train all reference regression models (`Linear Regression`, `Lasso`, `Ridge`, `KNN`, `SVM`, `Decision Tree`, `Random Forest`, `ExtraTrees`, `Bagging`, `AdaBoost`, `GradientBoosting`, `HistGradientBoosting`, plus `XGBoost`, `CatBoost`, and `LightGBM` when available), record train and test metrics, and plot actual vs. predicted `Performance Index` across the first 20 test samples.


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    'KNN': KNeighborsRegressor(n_neighbors=5, metric='minkowski', p=1),
    'SVM': SVR(kernel='linear', C=100, gamma=1),
    'Decision Tree': DecisionTreeRegressor(criterion='squared_error', max_depth=10),
    'Random Forest': RandomForestRegressor(n_estimators=50, criterion='squared_error', max_depth=10),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42),
    'Bagging': BaggingRegressor(estimator=DecisionTreeRegressor(criterion='squared_error', max_depth=10), n_estimators=50, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=15, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=100, learning_rate=0.1, random_state=42)
}
if HAS_XGB:
    models['XGBoost'] = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)
if HAS_CAT:
    models['CatBoost'] = CatBoostRegressor(verbose=0, iterations=100, learning_rate=0.1, random_state=42)
if HAS_LGBM:
    models['LightGBM'] = LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)


In [ ]:
y_train_1d = y_train.values.ravel()
y_test_1d = y_test.values.ravel()
errors = {}
for model_name, model in tqdm(models.items()):
    model.fit(X_train, y_train_1d)
    MAE_Train, MSE_Train, RMSE_Train, R_Train = regression_metrics(model, X_train, y_train_1d)
    MAE_Test, MSE_Test, RMSE_Test, R_Test = regression_metrics(model, X_test, y_test_1d)
    errors[model_name] = {
        'MAE Train': MAE_Train,
        'MAE Test': MAE_Test,
        'MSE Train': MSE_Train,
        'MSE Test': MSE_Test,
        'RMSE Train': RMSE_Train,
        'RMSE Test': RMSE_Test,
        'R2 Train': R_Train,
        'R2 Test': R_Test
    }
    y_test_plot = y_test_1d[:20]
    y_pred_plot = model.predict(X_test[:20])
    plt.figure(figsize=(10, 5))
    plt.plot(range(20), y_test_plot, label='Actual Performance Index', marker='o')
    plt.plot(range(20), y_pred_plot, label='Predicted Performance Index', marker='x')
    plt.xlabel('Sample Index')
    plt.ylabel('Performance Index')
    plt.title('Actual vs Predicted Performance Index (First 20 Samples) For ' + model_name)
    plt.legend()
    plt.grid()
    plt.show()


## 8. Voting Regression

Fit equal-weight and custom-weighted `VotingRegressor` ensembles (`Voting`, `Simple Voting`, and `Weighted Voting`) and record their metrics.


In [ ]:
clf1 = LinearRegression()
clf2 = RandomForestRegressor(n_estimators=50, random_state=1)
clf3 = DecisionTreeRegressor(random_state=1)
model1 = VotingRegressor(estimators=[('lr', clf1), ('rf', clf2), ('dt', clf3)])
model2 = VotingRegressor(estimators=[('lr', clf1), ('rf', clf2), ('dt', clf3)])
model3 = VotingRegressor(estimators=[('lr', clf1), ('rf', clf2), ('dt', clf3)], weights=[2, 1, 2])
model1.fit(X_train, y_train_1d)
model2.fit(X_train, y_train_1d)
model3.fit(X_train, y_train_1d)


In [ ]:
for v_name, v_model in [('Voting', model1), ('Simple Voting', model2), ('Weighted Voting', model3)]:
    MAE_Train, MSE_Train, RMSE_Train, R2_Train = regression_metrics(v_model, X_train, y_train_1d)
    MAE_Test, MSE_Test, RMSE_Test, R2_Test = regression_metrics(v_model, X_test, y_test_1d)
    errors[v_name] = {
        'MAE Train': MAE_Train,
        'MAE Test': MAE_Test,
        'MSE Train': MSE_Train,
        'MSE Test': MSE_Test,
        'RMSE Train': RMSE_Train,
        'RMSE Test': RMSE_Test,
        'R2 Train': R2_Train,
        'R2 Test': R2_Test
    }


## 9. Stacking Regression

Train a `StackingRegressor` combining ensemble base regressors with a meta-regressor (`passthrough=True`, `cv=5`) and record its metrics.


In [ ]:
base_regressors = [
    ('rf', RandomForestRegressor(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=50, learning_rate=0.1, random_state=42)),
    ('hgb', HistGradientBoostingRegressor(max_iter=50, learning_rate=0.1, random_state=42))
]
if HAS_CAT:
    base_regressors.append(('cat', CatBoostRegressor(iterations=50, learning_rate=0.1, verbose=0, random_state=42)))
if HAS_LGBM:
    base_regressors.append(('lgb', LGBMRegressor(n_estimators=50, learning_rate=0.1, random_state=42, verbose=-1)))
meta_regressor = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0) if HAS_XGB else GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
stacking_regressor = StackingRegressor(
    estimators=base_regressors,
    final_estimator=meta_regressor,
    cv=5,
    passthrough=True,
    n_jobs=1
)
stacking_regressor.fit(X_train, y_train_1d)
MAE_Train, MSE_Train, RMSE_Train, R2_Train = regression_metrics(stacking_regressor, X_train, y_train_1d)
MAE_Test, MSE_Test, RMSE_Test, R2_Test = regression_metrics(stacking_regressor, X_test, y_test_1d)
errors['Stacking'] = {
    'MAE Train': MAE_Train,
    'MAE Test': MAE_Test,
    'MSE Train': MSE_Train,
    'MSE Test': MSE_Test,
    'RMSE Train': RMSE_Train,
    'RMSE Test': RMSE_Test,
    'R2 Train': R2_Train,
    'R2 Test': R2_Test
}


## 10. Model Comparison Table and Metric Visualizations

Compile all models into `errors_df` sorted by `R2 Test` descending and visualize train vs. test metrics (`MAE`, `MSE`, `RMSE`, `R2`).


In [ ]:
errors_df = pd.DataFrame(errors).T.sort_values(by='R2 Test', ascending=False)
errors_df


In [ ]:
errors_df[['R2 Train', 'R2 Test']].plot.bar(figsize=(12, 6))
plt.title('R2 Score Comparison (Train vs Test)')
plt.ylabel('R2 Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
cols = [
    ['MAE Train', 'MAE Test'],
    ['MSE Train', 'MSE Test'],
    ['RMSE Train', 'RMSE Test'],
    ['R2 Train', 'R2 Test']
]
plt.figure(figsize=(20, 16))
for i, col in enumerate(cols):
    ax = plt.subplot(2, 2, i + 1)
    errors_df[col].plot.bar(ax=ax)
    ax.set_title(col[0].split(' ')[0] + ' Comparison')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## 11. Regression Reports for All Regression Models

Print MAE, MSE, RMSE, and R2 score for each evaluated regression model.


In [ ]:
all_models = dict(models)
all_models['Voting'] = model1
all_models['Simple Voting'] = model2
all_models['Weighted Voting'] = model3
all_models['Stacking'] = stacking_regressor
preds_dict = {name: model.predict(X_test) for name, model in all_models.items()}


In [ ]:
for name in preds_dict.keys():
    y_pred = preds_dict[name]
    mae = mean_absolute_error(y_test_1d, y_pred)
    mse = mean_squared_error(y_test_1d, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_1d, y_pred)
    print(f'=== {name} ===')
    print(f'MAE  : {mae:.4f}')
    print(f'MSE  : {mse:.4f}')
    print(f'RMSE : {rmse:.4f}')
    print(f'R2   : {r2:.6f}\n')


## 12. Final Conclusion

- **Problem Formulation:** `Student_Performance.csv` (`10,000` rows, `6` columns) is a **Regression** problem predicting continuous **`Performance Index`** (`Mean = 55.22`, `Median = 55.00`, range `10` to `100`), with missing values in `Sample Question Papers Practiced` (`2,856`) and `Extracurricular Activities` (`1,572`).
- **Feature Relationships:** `Previous Scores` (`r = 0.9152`) and `Hours Studied` (`r = 0.3737`) show strong positive correlation with `Performance Index`, whereas `Sleep Hours` (`r = 0.0481`) and `Sample Question Papers Practiced` (`r = 0.0369`) have weak direct linear correlation.
- **Model Performance Summary:** `SVM` (`R2 Test = 0.988213`, `MAE Test = 1.6485`, `RMSE Test = 2.0901`), `Linear Regression` (`R2 Test = 0.988204`), `Ridge` (`R2 Test = 0.988202`), and boosting/ensemble models (`HistGradientBoosting` `0.987375`, `GradientBoosting` `0.987278`, `Stacking` `0.986975`) achieve high predictive accuracy across the test split.
